In [ ]:
# | default_exp features.wps_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator, parse_array

log = structlog.get_logger()

In [ ]:
# | export


class WPSGenomeEvaluator(FeatureEvaluator):
    """Extracts genome-wide WPS metrics with spectral features.

    For each WPS array column and region_type, extracts:
    - mean, std (signal level and dispersion)
    - peak-to-valley amplitude (nucleosome occupancy proxy)

    Supports two extraction paths:
    - **SQL pushdown (preferred)**: DuckDB parses string arrays and computes
      aggregates entirely in C++ — orders of magnitude faster than Python.
    - **Python fallback**: Row-by-row extraction with parse_array() + numpy.
      Also computes MAD and FFT spectral features (not available in SQL).

    The SQL pushdown produces a "tall" result (one row per sample × region_type)
    that the CLI pivots to wide format via ``sql_pivot_column``.
    """

    name = "WPSGenome"
    source_file = ".WPS.parquet"
    tier = 2
    category = "epigenetics_and_geometry"
    # Column projection: only read the 5 columns used by extract().
    # WPS parquets have many additional columns — loading all of them
    # caused OOM on genome-wide runs (~30K rows x 50+ cols per sample).
    extract_columns = [
        "region_type",
        "wps_nuc",
        "wps_tf",
        "prot_frac_nuc",
        "prot_frac_tf",
    ]
    max_chunk_rows = 2_000_000  # Aggressive chunking for genomewide WPS data

    # When extract_sql() returns a multi-row-per-sample result, the CLI
    # pivots on this column to produce one row per sample with feature
    # columns named {pivot_value}_{stat}.
    sql_pivot_column = "region_type_clean"

    # Array column names used by both SQL and Python paths
    _array_cols = ("wps_nuc", "wps_tf", "prot_frac_nuc", "prot_frac_tf")

    def extract_sql(self) -> str | None:
        """DuckDB SQL pushdown for WPSGenome.

        Parses string-encoded arrays (e.g. ``"[1.0 2.0 3.0]"``) into DuckDB
        lists, then computes per-``region_type`` aggregate statistics entirely
        in DuckDB's C++ engine — eliminating the Python row loop.

        Computes per array column: mean, peak_valley (max - min).
        Skips: std, MAD, FFT spectral features (require numpy; minimal
        incremental AUC; WPSBackground already captures periodicity).

        Returns a tall DataFrame: one row per (sample_id, region_type).
        The CLI pivots this via ``sql_pivot_column`` to wide format.
        """
        # Build per-column aggregation expressions.
        # DuckDB parses "[1.0 2.0 3.0]" → DOUBLE[] via:
        #   string_split(trim(col, '[]'), ' ') → VARCHAR[]
        #   list_filter(... , x -> x != '') removes empty strings from double-spaces
        #   CAST(... AS DOUBLE[]) → typed numeric list
        agg_parts = []
        for col in self._array_cols:
            parse_expr = (
                f"CAST(list_filter("
                f"string_split(trim(trim({col}, '[]')), ' '), "
                f"x -> x != '' AND x IS NOT NULL"
                f") AS DOUBLE[])"
            )
            agg_parts.extend(
                [
                    f"list_avg({parse_expr}) AS {col}_mean",
                    f"(list_max({parse_expr}) - list_min({parse_expr})) AS {col}_peak_valley",
                ]
            )

        agg_clause = ",\n            ".join(agg_parts)

        return f"""
            SELECT
                regexp_extract(filename, '/([^/]+)/[^/]+$', 1) AS sample_id,
                replace(replace(replace(region_type, ' ', '_'), '-', '_'), '|', '_')
                    AS region_type_clean,
                {agg_clause}
            FROM read_parquet(
                ?,
                filename=true,
                union_by_name=true,
                hive_partitioning=false
            )
            WHERE region_type IS NOT NULL
        """

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        """Python fallback extraction — row-by-row with parse_array().

        Used when SQL pushdown is unavailable or returns empty. Computes
        all features including MAD and FFT spectral features.
        """
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "region_type" in cols:
                for row in df.to_dict("records"):
                    rt = (
                        str(row["region_type"])
                        .replace(" ", "_")
                        .replace("-", "_")
                        .replace("|", "_")
                    )
                    for a in self._array_cols:
                        if a in cols:
                            arr_raw = parse_array(row[a])
                            if arr_raw is not None and len(arr_raw) > 0:
                                arr = np.array(arr_raw)
                                extracted[f"{rt}_{a}_mean"] = float(np.mean(arr))
                                extracted[f"{rt}_{a}_std"] = float(np.std(arr))

                                extracted[f"{rt}_{a}_peak_valley"] = float(
                                    np.max(arr) - np.min(arr)
                                )
                                extracted[f"{rt}_{a}_mad"] = float(
                                    np.median(np.abs(arr - np.median(arr)))
                                )

                                if len(arr) >= 50:
                                    fft_vals = np.abs(np.fft.rfft(arr - arr.mean()))
                                    freqs = np.fft.rfftfreq(len(arr))
                                    if len(fft_vals) > 2:
                                        extracted[f"{rt}_{a}_spectral_max_power"] = (
                                            float(np.max(fft_vals[1:]))
                                        )
                                        extracted[
                                            f"{rt}_{a}_spectral_dominant_freq"
                                        ] = float(freqs[1:][np.argmax(fft_vals[1:])])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
from kreview.features.wps_genomewide import _parse_array

evaluator = WPSGenomeEvaluator()
df = pd.DataFrame([{"region_type": "Promo", "wps_nuc": "[1.0 1.0]"}])
res = evaluator.extract(df)
assert res["Promo_wps_nuc_mean"] == 1.0